# Notebook 02: Transformaciones & Silver/Gold Layers

## Objetivos
- Leer datos de la capa Bronze (Delta Lake)
- Limpieza y deduplicación
- Window functions: rolling aggregations por cuenta
- Feature engineering para detección de anomalías
- Broadcast joins con tablas de referencia
- Escritura Delta Silver + Gold
- Benchmark: AQE, caching, broadcast

In [1]:
import sys
sys.path.insert(0, '/home/jovyan')

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from src.config import (
    BRONZE_PATH, SILVER_PATH, GOLD_PATH, DATA_RAW,
    PARTITION_COLS_SILVER, SPARK_APP_NAME,
)
from src.utils import benchmark, setup_logging, show_table_info
from src.transformations import (
    build_silver_layer, add_rolling_account_stats,
    add_amount_zscore, enrich_with_merchants,
)
from src.fraud_rules import apply_all_fraud_rules, build_fraud_summary

setup_logging()

In [2]:
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName(f"{SPARK_APP_NAME}-silver-gold")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.sql.adaptive.skewJoin.enabled", "true")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print(f"Spark version: {spark.version}")

Spark version: 3.5.0


## 1. Lectura Bronze Layer

In [3]:
with benchmark("Lectura Bronze"):
    bronze_df = spark.read.format("delta").load(BRONZE_PATH)

show_table_info(bronze_df, "Bronze Layer")

2026-03-26 23:29:37 | src.utils | INFO | ⏱ Iniciando: Lectura Bronze
2026-03-26 23:29:42 | src.utils | INFO | ✅ Lectura Bronze completado en 4.77 segundos



📊 Bronze Layer
  Filas:    100,000
  Columnas: 16
  Particiones: 3

root
 |-- transaction_id: string (nullable = true)
 |-- account_id: string (nullable = true)
 |-- merchant_id: string (nullable = true)
 |-- transaction_timestamp: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- hour: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- merchant_category: string (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- country: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- is_fraud: integer (nullable = true)
 |-- risk_score: double (nullable = true)



## 2. Transformaciones Silver

Aplicamos: deduplicación, cast de tipos, filtro de inválidos, features temporales.

In [4]:
with benchmark("Transformaciones Silver (dedup + clean + time features)"):
    silver_df = build_silver_layer(bronze_df)
    silver_df.cache()
    silver_count = silver_df.count()

print(f"\nRegistros Silver: {silver_count:,}")
print(f"Registros eliminados: {bronze_df.count() - silver_count:,}")

2026-03-26 23:29:49 | src.utils | INFO | ⏱ Iniciando: Transformaciones Silver (dedup + clean + time features)
2026-03-26 23:29:52 | src.utils | INFO | ✅ Transformaciones Silver (dedup + clean + time features) completado en 3.95 segundos



Registros Silver: 100,000
Registros eliminados: 0


## 3. Feature Engineering

### 3a. Rolling Account Stats (Window Functions)

In [5]:
with benchmark("Rolling account stats (7-day window)"):
    silver_with_rolling = add_rolling_account_stats(silver_df, window_days=7)

silver_with_rolling.select(
    "account_id", "amount", "rolling_avg_amount",
    "rolling_count", "rolling_max_amount",
).show(10)

2026-03-26 23:29:53 | src.utils | INFO | ⏱ Iniciando: Rolling account stats (7-day window)
2026-03-26 23:29:53 | src.utils | INFO | ✅ Rolling account stats (7-day window) completado en 0.10 segundos


+------------+------+------------------+-------------+------------------+
|  account_id|amount|rolling_avg_amount|rolling_count|rolling_max_amount|
+------------+------+------------------+-------------+------------------+
|ACC_00000000| 18.77|             18.77|            1|             18.77|
|ACC_00000010|  3.87|              3.87|            1|              3.87|
|ACC_00000011| 49.35|             49.35|            1|             49.35|
|ACC_00000012|  9.93|              9.93|            1|              9.93|
|ACC_00000014|   3.3|               3.3|            1|               3.3|
|ACC_00000016|143.81|            143.81|            1|            143.81|
|ACC_00000018| 77.75|             77.75|            1|             77.75|
|ACC_00000019| 15.66|             15.66|            1|             15.66|
|ACC_00000020|  2.74|              2.74|            1|              2.74|
|ACC_00000021|  42.5|              42.5|            1|              42.5|
+------------+------+-----------------

### 3b. Z-Score de Montos

In [6]:
with benchmark("Cálculo de z-scores por cuenta"):
    silver_with_zscore = add_amount_zscore(silver_with_rolling)

silver_with_zscore.select(
    "account_id", "amount", "amount_zscore",
).where(F.abs(F.col("amount_zscore")) > 2).show(10)

2026-03-26 23:29:54 | src.utils | INFO | ⏱ Iniciando: Cálculo de z-scores por cuenta
2026-03-26 23:29:54 | src.utils | INFO | ✅ Cálculo de z-scores por cuenta completado en 0.13 segundos


+----------+------+-------------+
|account_id|amount|amount_zscore|
+----------+------+-------------+
+----------+------+-------------+



### 3c. Broadcast Join con Merchants

In [7]:
merchants_df = spark.read.parquet(str(DATA_RAW / "merchants"))
print(f"Merchants: {merchants_df.count():,} registros")

with benchmark("Broadcast join con merchants"):
    enriched_df = enrich_with_merchants(silver_with_zscore, merchants_df)

enriched_df.select(
    "transaction_id", "merchant_id", "merchant_name",
    "merchant_risk_level", "amount",
).show(5)

2026-03-26 23:29:55 | src.utils | INFO | ⏱ Iniciando: Broadcast join con merchants
2026-03-26 23:29:55 | src.utils | INFO | ✅ Broadcast join con merchants completado en 0.04 segundos


Merchants: 50,000 registros
+----------------+------------+--------------+-------------------+------+
|  transaction_id| merchant_id| merchant_name|merchant_risk_level|amount|
+----------------+------------+--------------+-------------------+------+
|TXN_000000000004|MER_00003940| Merchant_3940|               high| 13.55|
|TXN_000000000005|MER_00004268| Merchant_4268|                low| 23.41|
|TXN_000000000010|MER_00044323|Merchant_44323|                low|  3.87|
|TXN_000000000016|MER_00047010|Merchant_47010|             medium|143.81|
|TXN_000000000021|MER_00002159| Merchant_2159|                low|  42.5|
+----------------+------------+--------------+-------------------+------+
only showing top 5 rows



## 4. Escritura Silver Layer

In [8]:
with benchmark("Escritura Silver Layer (Delta + partición)"):
    (
        enriched_df
        .write
        .format("delta")
        .mode("overwrite")
        .partitionBy(*PARTITION_COLS_SILVER)
        .option("compression", "zstd")
        .save(SILVER_PATH)
    )

print(f"Silver layer guardada en: {SILVER_PATH}")

2026-03-26 23:29:56 | src.utils | INFO | ⏱ Iniciando: Escritura Silver Layer (Delta + partición)
2026-03-26 23:30:03 | src.utils | INFO | ✅ Escritura Silver Layer (Delta + partición) completado en 6.91 segundos


Silver layer guardada en: /home/jovyan/data/delta/silver/transactions


## 5. Reglas de Fraude y Gold Layer

In [9]:
# Leer Silver para aplicar reglas
silver_final = spark.read.format("delta").load(SILVER_PATH)

with benchmark("Aplicar todas las reglas de fraude"):
    fraud_df = apply_all_fraud_rules(silver_final)

# Distribución de fraud_score
fraud_df.groupBy(
    F.when(F.col("fraud_score") == 0, "0 (ningún flag)")
    .when(F.col("fraud_score") <= 0.3, "0.01-0.30 (bajo)")
    .when(F.col("fraud_score") <= 0.6, "0.31-0.60 (medio)")
    .otherwise(">0.60 (alto)")
    .alias("risk_bucket")
).count().orderBy("risk_bucket").show()

2026-03-26 23:30:03 | src.utils | INFO | ⏱ Iniciando: Aplicar todas las reglas de fraude
2026-03-26 23:30:03 | src.utils | INFO | ✅ Aplicar todas las reglas de fraude completado en 0.14 segundos


+----------------+-----+
|     risk_bucket|count|
+----------------+-----+
| 0 (ningún flag)|63121|
|0.01-0.30 (bajo)|36879|
+----------------+-----+



In [10]:
# Construir resumen Gold
with benchmark("Construcción Gold Layer (resumen de fraude)"):
    gold_df = build_fraud_summary(fraud_df)

    gold_df.write.format("delta").mode("overwrite").save(GOLD_PATH)

print(f"Gold layer guardada en: {GOLD_PATH}")
gold_df.show(10)

2026-03-26 23:30:09 | src.utils | INFO | ⏱ Iniciando: Construcción Gold Layer (resumen de fraude)
2026-03-26 23:30:14 | src.utils | INFO | ✅ Construcción Gold Layer (resumen de fraude) completado en 4.41 segundos


Gold layer guardada en: /home/jovyan/data/delta/gold/fraud_metrics
+-----------------+----+-----+------------------+-----------+----------+---------------+---------------+------------+----------+
|merchant_category|year|month|total_transactions|total_fraud|avg_amount|avg_fraud_score|high_risk_count|fraud_amount|fraud_rate|
+-----------------+----+-----+------------------+-----------+----------+---------------+---------------+------------+----------+
|       automotive|2024|    1|              1133|         14|     37.37|         0.0447|              0|     1426.11|  0.012357|
|         clothing|2024|    1|              1184|         10|     37.08|          0.046|              0|     1433.07|  0.008446|
|        education|2024|    1|              1216|         14|     37.06|         0.0438|              0|      1632.3|  0.011513|
|      electronics|2024|    1|              1144|         15|     38.03|         0.0455|              0|     2790.97|  0.013112|
|    entertainment|2024|    1|

In [11]:
# Liberar cache
silver_df.unpersist()
print("\n✅ Notebook 02 completado. Silver y Gold layers listas.")


✅ Notebook 02 completado. Silver y Gold layers listas.
